<a href="https://colab.research.google.com/github/Viggo-Kristensen/kaggle-competitions/blob/main/loan_payback_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Problem definition**

### Predict whether or not a borrower will pay back their loan.

## **Imports**

In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 15.5 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import optuna
import os

from sklearn.base import TransformerMixin, BaseEstimator
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance
from itertools import combinations, product

## **Dataset**

### Uploading Files

In [3]:
!mkdir -p /root/.kaggle
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

mv: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [4]:
# upload kaggle.json ONCE
from google.colab import files
files.upload()

!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle competitions download -c playground-series-s5e11

Saving kaggle.json to kaggle.json
100% 19.3M/19.3M [00:00<00:00, 29.9MB/s]



### Unzipping Files

In [5]:
!unzip -q playground-series-s5e11.zip -d data

### Creating dataframes

In [6]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

In [7]:
target = "loan_paid_back"
drop_cols = [target, "id"]
X = train.drop(columns=drop_cols)
y = train[target]

## **Feature Engineering**

In [8]:
class AddFeatures(BaseEstimator, TransformerMixin):
  def __init__(self, feature_pairs=None, operations=None):
    self.feature_pairs = feature_pairs
    self.operations = operations

  def fit(self, X, y=None):
    return self

  def transform(self, X):
    X = X.copy()

    if self.feature_pairs == None or self.operations == None:
      return X

    for f1, f2, op_name in self.feature_pairs:
      op = self.operations[op_name]
      X[f"{f1}_{op_name}_{f2}"] = op(X[f1], X[f2])

    return X

In [10]:
X_fe = AddFeatures().transform(X)

## **Preprocessing**

In [11]:
def build_preprocessor(X_sample):
  num_cols = X_sample.select_dtypes(include=["int64", "float64"]).columns
  cat_cols = X_sample.select_dtypes(include=["object", "bool"]).columns

  numerical_transformer = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median"))
  ])

  categorical_transformer = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
  ])

  preprocessor = ColumnTransformer(transformers=[
    ("cat", categorical_transformer, cat_cols),
    ("num", numerical_transformer, num_cols)
  ])

  return preprocessor

## **Pipeline**

In [12]:
def build_model(feature_pairs=None, operations=None, **best_params):
    X_tmp = AddFeatures(feature_pairs=feature_pairs, operations=operations).transform(X)

    preprocessor = build_preprocessor(X_tmp)

    return Pipeline(steps=[
        ("feature_engineering", AddFeatures(
            feature_pairs=feature_pairs,
            operations=operations
        )),
        ("preprocessing", preprocessor),
        ("classifier", XGBClassifier(
            **best_params,
            tree_method="hist",
            random_state=42
        ))
    ])

## **Hyperparameter Optimization**

In [22]:
def objective(trial):
  params = {
      "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
      "max_depth": trial.suggest_int("max_depth", 3, 12),
      "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
      "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
      "min_child_weight": trial.suggest_int("min_child_weight", 0, 10),
      "subsample": trial.suggest_float("subsample", 0.6, 1.0),
      "reg_lambda": trial.suggest_float("reg_lambda", 0, 10),
      "reg_alpha": trial.suggest_float("reg_alpha", 0, 10),
      "gamma": trial.suggest_float("gamma", 0, 10),
    }

  model = build_model(**params)

  scores = cross_val_score(
      model,
      X,
      y,
      cv=5,
      scoring="roc_auc",
      n_jobs=-1
  )

  return scores.mean()

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=5)

best_params = study.best_params
best_value = study.best_value

print("best params: ", best_params)
print("best value: ", best_value)

### Optuna Result Visualization

Optimization history plot

In [ ]:
vis.plot_optimization_history(study)

Hyperparameter importance plot

In [ ]:
vis.plot_param_importances(study)

Parallel coordinate plot

In [ ]:
vis.plot_parallel_coordinate(study)

## **Best Hyperparameters**

In [13]:
best_params = {
    'n_estimators': 1166,
    'max_depth': 4,
    'learning_rate': 0.07792559951270059,
    'colsample_bytree': 0.7579610416258149,
    'min_child_weight': 2,
    'subsample': 0.9639464946900222,
    'reg_lambda': 5.620182425302145,
    'reg_alpha': 3.663398453360321,
    'gamma': 0.43267842976701226}


## **Best Features**

In [20]:
best_features = [
    ("debt_to_income_ratio", "credit_score", "sub"),
    ("credit_score", "loan_amount", "add"),
    ("credit_score", "loan_amount", "sub")
]

## **Model**

In [21]:
model = build_model(feature_pairs=best_features, **best_params)

## **Feature Engineering Validation**

### Creating features to test

In [22]:
operations = {
    "mul": lambda a, b: a * b,
    "add": lambda a, b: a + b,
    "sub": lambda a, b: a - b,
    "div": lambda a, b: a / (b + 1e-6)
}

top_features = [
    "debt_to_income_ratio",
    "credit_score",
    "loan_amount"
]

feature_pairs = [
    (f1, f2, op)
    for f1, f2 in combinations(top_features, 2)
    for op in operations.keys()
]

### Feature engineering validation loop

In [18]:
def evaluate_feature_engineering():
    baseline_model = build_model(feature_pairs=None, operations=operations, **best_params)

    baseline_score = cross_val_score(
        baseline_model,
        X,
        y,
        cv=5,
        scoring="roc_auc",
        n_jobs=-1
    ).mean()

    print("Baseline:", baseline_score)

    best_score = baseline_score
    best_features = None

    for f1, f2, op in feature_pairs:
        test_pairs = [(f1, f2, op)]

        model = build_model(feature_pairs=test_pairs, operations=operations, **best_params)

        score = cross_val_score(
            model,
            X,
            y,
            cv=5,
            scoring="roc_auc",
            n_jobs=-1
        ).mean()

        print(f"{f1}-{op}-{f2}: {score}")

        if score > best_score:
            best_score = score
            best_features = test_pairs

    return best_features, best_score

best_features, best_score = evaluate_feature_engineering()

Baseline: 0.9236636577770682
debt_to_income_ratio-mul-credit_score: 0.9236383460672627
debt_to_income_ratio-add-credit_score: 0.9236538394890973
debt_to_income_ratio-sub-credit_score: 0.9236874854541032
debt_to_income_ratio-div-credit_score: 0.9235970947172015
debt_to_income_ratio-mul-loan_amount: 0.9235927620437085
debt_to_income_ratio-add-loan_amount: 0.9236233664002664
debt_to_income_ratio-sub-loan_amount: 0.9236623935221335
debt_to_income_ratio-div-loan_amount: 0.923546308697113
credit_score-mul-loan_amount: 0.9236000561616692
credit_score-add-loan_amount: 0.923691301195063
credit_score-sub-loan_amount: 0.9236831839867413
credit_score-div-loan_amount: 0.9235926209011165


([('credit_score', 'loan_amount', 'add')], np.float64(0.923691301195063))

## **Training**

In [23]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)

Pipeline(steps=[('feature_engineering',
                 AddFeatures(feature_pairs=[('debt_to_income_ratio',
                                             'credit_score', 'sub'),
                                            ('credit_score', 'loan_amount',
                                             'add'),
                                            ('credit_score', 'loan_amount',
                                             'sub')])),
                ('preprocessing',
                 ColumnTransformer(transformers=[('cat',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown...
                               gamma=0.43267842976701226, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None,
                               learning_rate=0.07792559951270059, max_bin=None,
                               max_cat_threshold=None, max_cat_to_onehot=None,
                               max_delta_step=None, max_depth=4,
                               max_leaves=None, min_child_weight=2, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=1166, n_jobs=None,
                               num_parallel_tree=None, ...))])

## **Permutation Importance**

In [ ]:
result = permutation_importance(
    model,
    X_val,
    y_val,
    n_repeats=3,
    scoring="roc_auc"
)

perm_df = pd.DataFrame({
    "feature": X_val.columns,
    "importance": result.importances_mean
}).sort_values("importance", ascending=False)

print(perm_df)

## **Evaluation**

In [24]:
y_probs = model.predict_proba(X_val)[:, 1]
roc_auc_score(y_val, y_probs)

np.float64(0.9245368321705107)

## **Final Training**

In [25]:
model.fit(X, y)

Pipeline(steps=[('feature_engineering',
                 AddFeatures(feature_pairs=[('debt_to_income_ratio',
                                             'credit_score', 'sub'),
                                            ('credit_score', 'loan_amount',
                                             'add'),
                                            ('credit_score', 'loan_amount',
                                             'sub')])),
                ('preprocessing',
                 ColumnTransformer(transformers=[('cat',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown...
                               gamma=0.43267842976701226, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None,
                               learning_rate=0.07792559951270059, max_bin=None,
                               max_cat_threshold=None, max_cat_to_onehot=None,
                               max_delta_step=None, max_depth=4,
                               max_leaves=None, min_child_weight=2, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=1166, n_jobs=None,
                               num_parallel_tree=None, ...))])

## **Submission**

In [29]:
submission = pd.read_csv("data/sample_submission.csv")
submission["loan_paid_back"] = model.predict_proba(test)[:, 1]
submission.to_csv("submission.csv", index=False)

## **Reflections**

**Notes**
- Through the permutation importance analysis it has been deducted that `debt_to_income_ration`, `credit_score` and `loan_amount` were the most useful and predictive features for the model.
- Feature engineering choices were found through a bruteforce validation loop trying different operations between feature pairs created from the 3 best performing in the permutation importance analysis.
- The best hyperparameter choices were found through a study done using the optuna library.


**What i learned?**
- how to create a brute force feature engineering testing loop.
- How to open up pipelines to make custom choices.

